# Day 02: Nonlinear State Estimation & Visual FAST-Kalman Tracking
**State Estimation and Localization for Self-Driving Cars**

Today we tackle nonlinear dynamic systems and computer vision-integrated tracking:
1. **Analytical Jacobians**: Systematic derivation of $\mathbf{F}_{k-1}, \mathbf{L}_{k-1}, \mathbf{H}_k, \mathbf{M}_k$.
2. **The Extended Kalman Filter (EKF)**: Landmark bearing and 2D radar tracking.
3. **The Unscented Kalman Filter (UKF)**: Deterministic sampling with the Scaled Unscented Transform.
4. **Visual Motion Prediction with FAST Algorithm & Kalman Filter**:
   - Extraction of FAST corner keypoints and ORB descriptors on Region of Interest (ROI).
   - Frame-by-frame brute-force feature matching and centroid computation.
   - Forward motion prediction and Kalman correction.


In [ ]:
import cv2
import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from position_class import (
    ExtendedKalmanFilter,
    LandmarkBearingEKF,
    Radar2DTargetTrackerEKF,
    UnscentedKalmanFilter,
    LandmarkBearingUKF,
    FastKalmanVisualTracker,
    generate_synthetic_tracking_video,
    wraptopi
)

print("Day 02 Modules Loaded Successfully!")


## 1. Landmark Bearing Tracking: EKF vs UKF

Consider a vehicle observing the relative bearing $\theta_k$ to a stationary landmark at $\mathbf{l} = [x_l, y_l]^T$:

$$\mathbf{x}_k = \begin{bmatrix} x_k \\ y_k \\ \theta_k \end{bmatrix}, \quad
h(\mathbf{x}_k) = \operatorname{atan2}(y_l - y_k, x_l - x_k) - \theta_k$$

Measurement Jacobian $\mathbf{H}_k$:

$$\mathbf{H}_k = \left[ \frac{y_l - y_k}{d^2}, \quad -\frac{x_l - x_k}{d^2}, \quad -1 \right], \quad d^2 = (x_l - x_k)^2 + (y_l - y_k)^2$$


In [ ]:
# Simulation of Optical Landmark Bearing Navigation (Coursera / Excersice.ipynb)
np.random.seed(42)
dt = 0.5
steps = 60
t = np.linspace(0, steps * dt, steps)
S = 20.0  # Landmark lateral distance (m)
D = 40.0  # Landmark longitudinal position (m)

# True motion: constant velocity vehicle moving along x-axis from p=0 to p=30
true_p = 5.0 * t
true_v = np.full_like(t, 5.0)

# True bearing angle measurements
true_bearing = np.arctan(S / (D - true_p))
meas_bearing = true_bearing + np.random.normal(0, np.sqrt(0.01), steps)

# Initialize EKF and UKF
ekf = LandmarkBearingEKF(dt=dt, S=S, D=D, R=0.01)
ukf = LandmarkBearingUKF(dt=dt, S=S, D=D, R=0.01)

ekf_pos, ukf_pos = [], []
ekf_cov, ukf_cov = [], []

for k in range(steps):
    x_ekf, P_ekf = ekf.step(u=0.0, y=meas_bearing[k])
    x_ukf, P_ukf = ukf.step(u=0.0, y=meas_bearing[k])
    
    ekf_pos.append(x_ekf[0, 0])
    ukf_pos.append(x_ukf[0, 0])
    ekf_cov.append(P_ekf[0, 0])
    ukf_cov.append(P_ukf[0, 0])

ekf_pos = np.array(ekf_pos)
ukf_pos = np.array(ukf_pos)

# Plot EKF vs UKF
fig = make_subplots(rows=2, cols=1, subplot_titles=("<b>Vehicle Position: Ground Truth vs EKF vs UKF</b>", "<b>Bearing Angle Measurements & Nonlinear Model</b>"))
fig.add_trace(go.Scatter(x=t, y=true_p, mode='lines', name='Ground Truth Position', line=dict(color='black', width=3)), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=ekf_pos, mode='lines', name='EKF Estimate', line=dict(color='blue', dash='dash')), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=ukf_pos, mode='lines', name='UKF Estimate', line=dict(color='orange', dash='dot', width=2)), row=1, col=1)

fig.add_trace(go.Scatter(x=t, y=np.degrees(true_bearing), mode='lines', name='True Bearing (deg)', line=dict(color='black')), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=np.degrees(meas_bearing), mode='markers', name='Noisy Bearing Measurements', marker=dict(color='red', size=4)), row=2, col=1)

fig.update_layout(
    title='<b>Nonlinear Landmark Navigation: EKF vs UKF Performance</b>',
    height=650
)
fig.show()


## 2. Visual Object Motion Prediction: FAST Algorithm + Kalman Filter

Inspired by the VisionBrick visual motion tracking architecture:
- **FAST (Features from Accelerated Segment Test)** provides high-speed corner extraction in ROI.
- **ORB/BRIEF Descriptors** provide scale- and rotation-invariant feature representations.
- **Kalman Filter** provides constant-velocity motion prediction $\hat{\mathbf{x}}_{k|k-1} = \mathbf{F} \hat{\mathbf{x}}_{k-1|k-1}$ and continuous smoothing against noisy visual detections and occlusions.


In [ ]:
# Generate synthetic video stream with textured moving object
frames, gt_centroids = generate_synthetic_tracking_video(num_frames=80, width=640, height=480)

# Initialize FAST + Kalman Visual Tracker on Frame 0
tracker = FastKalmanVisualTracker(dt=1.0/30.0, process_noise_std=1.0, measurement_noise_std=2.0, fast_threshold=15)
initial_bbox = (int(gt_centroids[0][0] - 20), int(gt_centroids[0][1] - 20), 40, 40)
num_kps = tracker.initialize_target_from_roi(frames[0], initial_bbox)
print(f"Target ROI initialized with {num_kps} FAST keypoints.")

# Run tracking pipeline over frame sequence
tracked_results = []
for idx, frame in enumerate(frames):
    res = tracker.process_frame(frame)
    tracked_results.append(res)

gt_arr = np.array(gt_centroids)
pred_x = [r["pred_x"] for r in tracked_results]
pred_y = [r["pred_y"] for r in tracked_results]
meas_x = [r["meas_x"] if r["meas_x"] is not None else np.nan for r in tracked_results]
meas_y = [r["meas_y"] if r["meas_y"] is not None else np.nan for r in tracked_results]
est_x = [r["est_x"] for r in tracked_results]
est_y = [r["est_y"] for r in tracked_results]
matches = [r["matches"] for r in tracked_results]

# Plot Visual Tracking Performance
fig = make_subplots(rows=2, cols=1, subplot_titles=("<b>Visual Object Tracking: Ground Truth vs FAST Matches vs Kalman Estimates</b>", "<b>Number of Matched FAST Keypoints per Frame</b>"))

fig.add_trace(go.Scatter(x=gt_arr[:, 0], y=gt_arr[:, 1], mode='lines', name='Ground Truth Centroid', line=dict(color='black', width=3)), row=1, col=1)
fig.add_trace(go.Scatter(x=meas_x, y=meas_y, mode='markers', name='FAST Matched Centroid', marker=dict(color='red', size=6, symbol='x')), row=1, col=1)
fig.add_trace(go.Scatter(x=est_x, y=est_y, mode='lines', name='Kalman Filtered Position', line=dict(color='cyan', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=pred_x, y=pred_y, mode='lines', name='Kalman Predicted Position', line=dict(color='gold', dash='dot')), row=1, col=1)

fig.add_trace(go.Bar(x=list(range(len(matches))), y=matches, name='Matched Features', marker=dict(color='royalblue')), row=2, col=1)

fig.update_layout(height=750, title_text="<b>VisionBrick FAST + Kalman Filter Motion Prediction System</b>")
fig.show()
